In [1]:
library(fixest)
library(dplyr)
library(tidyr)
library(ggplot2)
library(lubridate)
library(modelsummary)

panel <- read.csv("../data/monthly_panel_mttu_mttr.csv")
raw_df <- read.csv("../data/sc_control_mttu_mttr_data_final_with_agg_score_and_vuln_count.csv")

raw_df <- raw_df %>%
    mutate(year_month = floor_date(as_date(published_at), "month"))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




## **Without Categorizing the Security Checks**

## MTTU

In [2]:
metrics <- c(
    "Binary_Artifacts_score", "Code_Review_score",
    "Contrib_score", "Dangerous_Workflow_score", "DependUpTool_score",
    "Fuzzing_score", "License_score",
    "Maintained_score", "Pinned_Dependencies_score",
    "Security_Policy_score", "Token_Permissions_score"
)

controls <- "log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) + version_age_days + log1p(Release_count)"
fe_part  <- "package_name + year_month"

metrics_joined <- paste(metrics, collapse = " + ")

## Build formula: continuous versions for all the metrics
# Aggregate_score is excluded from this model entirely.
fml_str <- paste("MTTU ~", metrics_joined, "+", controls, "|", fe_part)
fml     <- as.formula(fml_str)

## Run the model
model_all <- feglm(fml, data = panel, family = "poisson", cluster = ~package_name)

summary(model_all)

NOTES: 11,290 observations removed because of NA values (RHS: 11,290).
       4,415/0 fixed-effects (13,567 observations) removed because of only 0 outcomes or singletons.



GLM estimation, family = poisson, Dep. Var.: MTTU
Observations: 54,206
Fixed-effects: package_name: 7,220,  year_month: 25
Standard-errors: Clustered (package_name) 
                              Estimate Std. Error    z value   Pr(>|z|)    
Binary_Artifacts_score       -0.007397   0.030646  -0.241369 8.0927e-01    
Code_Review_score             0.005353   0.004081   1.311840 1.8957e-01    
Contrib_score                -0.004390   0.014094  -0.311490 7.5543e-01    
Dangerous_Workflow_score      0.001922   0.005548   0.346510 7.2896e-01    
DependUpTool_score           -0.001293   0.005073  -0.254889 7.9881e-01    
Fuzzing_score                -0.003987   0.008052  -0.495134 6.2051e-01    
License_score                -0.009885   0.023556  -0.419648 6.7474e-01    
Maintained_score             -0.037495   0.003358 -11.164480  < 2.2e-16 ***
Pinned_Dependencies_score    -0.001848   0.007892  -0.234151 8.1487e-01    
Security_Policy_score        -0.015663   0.006224  -2.516350 1.1858e-02 * 

## MTTR 

In [3]:
metrics <- c(
    "Binary_Artifacts_score", "Code_Review_score",
    "Contrib_score", "Dangerous_Workflow_score", "DependUpTool_score",
    "Fuzzing_score", "License_score",
    "Maintained_score", "Pinned_Dependencies_score",
    "Security_Policy_score", "Token_Permissions_score"
)

controls <- "log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) + version_age_days + log1p(Release_count)"
fe_part  <- "package_name + year_month"

metrics_joined <- paste(metrics, collapse = " + ")

## Build formula: continuous versions for all the metrics
# Aggregate_score is excluded from this model entirely.
fml_str <- paste("MTTR ~", metrics_joined, "+", controls, "|", fe_part)
fml     <- as.formula(fml_str)

## Run the model
model_all <- feglm(fml, data = panel, family = "poisson", cluster = ~package_name)

summary(model_all)

NOTES: 11,290 observations removed because of NA values (RHS: 11,290).
       10,339/0 fixed-effects (55,343 observations) removed because of only 0 outcomes or singletons.



GLM estimation, family = poisson, Dep. Var.: MTTR
Observations: 12,430
Fixed-effects: package_name: 1,296,  year_month: 25
Standard-errors: Clustered (package_name) 
                              Estimate Std. Error   z value   Pr(>|z|)    
Binary_Artifacts_score       -0.017134   0.066956 -0.255904 7.9803e-01    
Code_Review_score             0.008303   0.007641  1.086634 2.7720e-01    
Contrib_score                 0.000219   0.018129  0.012103 9.9034e-01    
Dangerous_Workflow_score     -0.008313   0.013452 -0.618003 5.3657e-01    
DependUpTool_score            0.006548   0.006426  1.018906 3.0825e-01    
Fuzzing_score                 0.035137   0.013079  2.686522 7.2200e-03 ** 
License_score                 0.033168   0.009464  3.504637 4.5723e-04 ***
Maintained_score             -0.025176   0.006075 -4.144173 3.4104e-05 ***
Pinned_Dependencies_score    -0.024733   0.019935 -1.240667 2.1473e-01    
Security_Policy_score        -0.023325   0.026045 -0.895547 3.7049e-01    
Token_Per

## **With Categorized Security Checks**

## MTTU

In [4]:
# Dichotomize variables 
## worthwile to rerun the tests with x == 0, etc to check sensitivity to the tolerance choice 

bin_zero_pos <- function(x) {              # adoption: 0 vs any positive
  factor(if_else(x < 0.5, "zero", "positive"),
         levels = c("zero", "positive"))
}

bin_ten_less <- function(x) {              # regression: full marks vs not
  factor(if_else(x > 9.5, "ten", "below_ten"),
         levels = c("below_ten", "ten"))
}


binnings <- list(zero_pos = bin_zero_pos, ten_less = bin_ten_less)

chosen_binning <- c(
  # Aggregate_score            = "continuous",
  Maintained_score           = "continuous",
  Code_Review_score          = "continuous",
  Pinned_Dependencies_score  = "zero_pos",
  Security_Policy_score      = "zero_pos",
  Token_Permissions_score    = "zero_pos",
  DependUpTool_score         = "zero_pos",
  Binary_Artifacts_score     = "ten_less",
  Dangerous_Workflow_score   = "ten_less",
  Contrib_score              = "ten_less"
)

binned_metrics <- chosen_binning[chosen_binning != "continuous"]
continuous_metrics <- names(chosen_binning[chosen_binning == "continuous"])

# 1. Create binned columns in panel
panel_binned <- panel
for (m in names(binned_metrics)) {
  f <- binnings[[binned_metrics[[m]]]]
  panel_binned[[paste0(m, "_bin")]] <- f(panel_binned[[m]])
}

# 2. Build formula: binned versions for the binned metrics,
# raw continuous columns for Maintained_score and Code_Review_score
# Aggregate_score is excluded from this model 
bin_terms  <- paste(paste0(names(binned_metrics), "_bin"), collapse = " + ")
cont_terms <- paste(continuous_metrics, collapse = " + ")

fml_all_binned_str <- paste(
  "MTTU ~",
  paste(c(cont_terms, bin_terms), collapse = " + "), "+",
  controls,
  "|", fe_part
)
fml_all_binned <- as.formula(fml_all_binned_str)

# 3. Run the model
model_all_binned <- feglm(fml_all_binned, data = panel_binned,
                           family = "poisson", cluster = ~package_name)
summary(model_all_binned)

NOTES: 11,248 observations removed because of NA values (RHS: 11,248).
       4,414/0 fixed-effects (13,575 observations) removed because of only 0 outcomes or singletons.



GLM estimation, family = poisson, Dep. Var.: MTTU
Observations: 54,240
Fixed-effects: package_name: 7,224,  year_month: 25
Standard-errors: Clustered (package_name) 
                                       Estimate Std. Error    z value
Maintained_score                      -0.037373   0.003355 -11.138211
Code_Review_score                      0.005195   0.004080   1.273322
Pinned_Dependencies_score_binpositive -0.041283   0.048342  -0.853981
Security_Policy_score_binpositive     -0.136418   0.059024  -2.311240
Token_Permissions_score_binpositive    0.007998   0.051485   0.155338
DependUpTool_score_binpositive        -0.021063   0.049296  -0.427277
Binary_Artifacts_score_binten         -0.082960   0.072057  -1.151313
Dangerous_Workflow_score_binten        0.015621   0.053782   0.290460
Contrib_score_binten                  -0.002755   0.083584  -0.032965
log1p(downloads_7_day_total)          -0.006595   0.003671  -1.796339
log1p(dependents)                      0.019471   0.010243   1.9

## MTTR

In [5]:
# Dichotomize variables 
## worthwile to rerun the tests with x == 0, etc to check sensitivity to the tolerance choice 

bin_zero_pos <- function(x) {              # adoption: 0 vs any positive
  factor(if_else(x < 0.5, "zero", "positive"),
         levels = c("zero", "positive"))
}

bin_ten_less <- function(x) {              # regression: full marks vs not
  factor(if_else(x > 9.5, "ten", "below_ten"),
         levels = c("below_ten", "ten"))
}


binnings <- list(zero_pos = bin_zero_pos, ten_less = bin_ten_less)

chosen_binning <- c(
  # Aggregate_score            = "continuous",
  Maintained_score           = "continuous",
  Code_Review_score          = "continuous",
  Pinned_Dependencies_score  = "zero_pos",
  Security_Policy_score      = "zero_pos",
  Token_Permissions_score    = "zero_pos",
  DependUpTool_score         = "zero_pos",
  Binary_Artifacts_score     = "ten_less",
  Dangerous_Workflow_score   = "ten_less",
  Contrib_score              = "ten_less"
)

binned_metrics <- chosen_binning[chosen_binning != "continuous"]
continuous_metrics <- names(chosen_binning[chosen_binning == "continuous"])

# 1. Create binned columns in panel
panel_binned <- panel
for (m in names(binned_metrics)) {
  f <- binnings[[binned_metrics[[m]]]]
  panel_binned[[paste0(m, "_bin")]] <- f(panel_binned[[m]])
}

# 2. Build formula: binned versions for the binned metrics,
# raw continuous columns for Maintained_score and Code_Review_score
# Aggregate_score is excluded from this model 
bin_terms  <- paste(paste0(names(binned_metrics), "_bin"), collapse = " + ")
cont_terms <- paste(continuous_metrics, collapse = " + ")

fml_all_binned_str <- paste(
  "MTTR ~",
  paste(c(cont_terms, bin_terms), collapse = " + "), "+",
  controls,
  "|", fe_part
)
fml_all_binned <- as.formula(fml_all_binned_str)

# 3. Run the model
model_all_binned <- feglm(fml_all_binned, data = panel_binned,
                           family = "poisson", cluster = ~package_name)
summary(model_all_binned)

NOTES: 11,248 observations removed because of NA values (RHS: 11,248).
       10,342/0 fixed-effects (55,382 observations) removed because of only 0 outcomes or singletons.



GLM estimation, family = poisson, Dep. Var.: MTTR
Observations: 12,433
Fixed-effects: package_name: 1,296,  year_month: 25
Standard-errors: Clustered (package_name) 
                                       Estimate Std. Error   z value   Pr(>|z|)
Maintained_score                      -0.024702   0.006071 -4.068952 4.7225e-05
Code_Review_score                      0.008229   0.007673  1.072474 2.8351e-01
Pinned_Dependencies_score_binpositive -0.036400   0.056217 -0.647481 5.1732e-01
Security_Policy_score_binpositive     -0.288999   0.207792 -1.390806 1.6428e-01
Token_Permissions_score_binpositive   -0.227303   0.139945 -1.624232 1.0433e-01
DependUpTool_score_binpositive         0.052554   0.063346  0.829633 4.0675e-01
Binary_Artifacts_score_binten         -0.199795   0.260455 -0.767101 4.4302e-01
Dangerous_Workflow_score_binten       -0.086819   0.134633 -0.644858 5.1902e-01
Contrib_score_binten                  -0.040096   0.120061 -0.333962 7.3841e-01
log1p(downloads_7_day_total)      